<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/Riffs3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Riff 3

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

Cloning into 'DS5001-Final-Project'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 247 (delta 37), reused 4 (delta 4), pack-reused 175 (from 1)
Receiving objects: 100% (247/247), 14.89 MiB | 6.46 MiB/s, done.
Resolving deltas: 100% (88/88), done.


In [2]:
!wget -O CORPUS.csv "https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv"
!wget -O LIB.csv "https://virginia.box.com/shared/static/fhzudg34je9xls5bfcbi4xdnaiek74rj.csv"
!wget -O TSNE.csv "https://virginia.box.com/shared/static/rquke7qx0ne3jvvyp8512h6rqjjtg81c.csv"
!wget -O THETA.csv "https://virginia.box.com/shared/static/xap24wuixe7l1gm0xnhywtddmfmij38e.csv"

--2026-04-21 00:48:12--  https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.box.com (virginia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.box.com (virginia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 00:48:12--  https://virginia.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Reusing existing connection to virginia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 00:48:12--  https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.app.box.com (virginia.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.app.box.com (virginia.app.box.com)|74.112.186.157|:443.

In [3]:
import pandas as pd
CORPUS = pd.read_csv('CORPUS.csv', delimiter='|', index_col=[0,1,2,3])
LIB = pd.read_csv('LIB.csv', delimiter='|', index_col=0)
TSNE = pd.read_csv('TSNE.csv', delimiter='|', index_col=0)
THETA = pd.read_csv('THETA.csv', delimiter='|', index_col=[0,1,2])

In [4]:
! pip install kaleido==0.2.1

##Taylor Swift Topics by Album

In [12]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'
import kaleido

TOPIC_NAMES = {
    'T18': 'Introspection',
    'T02': 'Daily Life',
    'T07': 'Longing',
    'T13': 'Romance',
    'T06': 'Ad-libs',
}
TS_ALBUMS = [
    'Taylor Swift',
    'Fearless',
    'Speak Now',
    'Red (Deluxe Edition)',
    '1989',
    'reputation',
    'Lover',
    'folklore',
    'evermore',
]

ts_theta = THETA[list(TOPIC_NAMES.keys())].copy()
ts_theta.index = pd.MultiIndex.from_tuples(THETA.index, names=['Artist', 'Album', 'Title'])
ts_theta = ts_theta.reset_index()
ts_theta = ts_theta[ts_theta['Artist'] == 'Taylor Swift']

album_year = (LIB[LIB['Artist'] == 'Taylor Swift']
              .groupby('Album')['Year'].first())

ts_theta['Year'] = ts_theta['Album'].map(album_year)

ts_theta = ts_theta.rename(columns=TOPIC_NAMES)
ts_theta = ts_theta[ts_theta['Album'].isin(TS_ALBUMS)]
album_topics = (ts_theta.groupby(['Album', 'Year'])[list(TOPIC_NAMES.values())]
                .mean()
                .reset_index()
                .sort_values('Year'))

album_topics['Album'] = [f"{a} ({int(y)})" for a, y in zip(album_topics['Album'], album_topics['Year'])]

plot_df = album_topics.melt(
    id_vars='Album',
    value_vars=list(TOPIC_NAMES.values()),
    var_name='Topic',
    value_name='Weight'
)

album_order = album_topics['Album'].tolist()

fig = px.bar(
    plot_df,
    x='Album',
    y='Weight',
    color='Topic',
    category_orders={'Album': album_order},
    title='Taylor Swift Topic Composition by Album',
    labels={'Weight': 'Mean Topic Weight', 'Album': ''},
    color_discrete_sequence=px.colors.qualitative.Pastel,
    height=600,
    width=1100
)

fig.update_layout(
    barmode='stack',
    xaxis_tickangle=-35,
    legend_title='Topic',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

import time
time.sleep(2)
pio.write_image(fig, 'taylorswift_albums.png', format='png', scale=2, engine='kaleido')
fig.show()